In [ ]:
# SPEC §11 (decision 2026-10-27): E1's second wave runs here while E5 has the laptop.
# Needs MyDrive/mtp/ur5rl_bundle.zip (rebuilt 2026-10-27, carries configs/exp/e1.yaml). CPU runtime.
# One run at a time (Colab's 2 cores fit one run; ~80 min per 500k-step run at ~105 steps/s). Checkpoints go to
# MyDrive/mtp/runs every 25k steps; finished runs are skipped, so after a disconnect just run this cell again.
# To split with a second machine (e.g. Kaggle), give each machine its own share of JOBS.
JOBS = [('td3', 2), ('sac', 1), ('sac', 2)]
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
D, W = '/content/drive/MyDrive/mtp', '/content/ur5rl'
sh = lambda c: subprocess.run(c, shell=True, check=True)
if not os.path.exists(f'{W}/scripts/train.py'):  # fresh VM: unpack and install once
    sh(f'mkdir -p {W} && unzip -q -o {D}/ur5rl_bundle.zip -d {W}')
    sh('pip install -q --index-url https://download.pytorch.org/whl/cpu torch==2.14.1')
    sh('pip install -q mujoco==3.17.0 gymnasium==1.4.0 stable-baselines3==2.9.2 numpy==2.4.6 scipy==1.17.1 '
       'PyYAML==6.0.3 psutil==7.2.2 tensorboard==2.21.0')
    sh(f'pip install -q -e {W} --no-deps --ignore-requires-python')
for run, seed in JOBS:
    print(f'=== E1 {run} seed {seed}', flush=True)
    sh(f'cd {W} && python scripts/train.py --exp configs/exp/e1.yaml --run {run} --seed {seed} '
       f'--threads 2 --out {D}/runs --ckpt-every 25000 2>&1 | grep --line-buffered -E "val]|curriculum|resumed|done|Error|Traceback"')
print('finished; copy', f'{D}/runs/e1/*', 'into results/runs/e1 on the laptop')
